## 10.2 CNN综合案例 - 模型构建与训练

#### 1. 案例目标

##### 1.1 本小节要完成什么
这一小节我们不再把数据、模型、训练拆成很多小节，

而是直接把一个 完整的 CNN 图像分类流程 串起来，形成一个可以独立运行的综合案例。🚀

这一节会完整包含：
* 数据下载
* 数据集拆分
* 图像预处理
* DataLoader 构建
* CNN 模型搭建
* 损失函数与优化器
* scheduler 学习率衰减
* 训练与验证
* 最终测试评估

从 0 到 1，完整走一遍 CIFAR-10 的 CNN 分类流程。

##### 1.2 本案例中会练到哪些旧知识
这个综合案例会把我们前面学过的很多内容都串起来，包括：
* CIFAR10 数据集读取
* transforms.Compose() 图像预处理
* random_split() 划分训练集和验证集
* DataLoader 组成 batch
* 卷积层、激活函数、池化层
* Flatten + Linear 分类头
* Dropout
* CrossEntropyLoss
* Adam 或 SGD
* scheduler 学习率衰减
* train / val / test 三阶段流程

#### 2 整体流程先建立印象
##### 💡 完整流程图
这个案例的整体流程可以先概括为：

下载 CIFAR-10

→ 配置 transforms

→ 构建 train / val / test 数据集

→ DataLoader

→ CNN 模型

→ 损失函数 + 优化器 + scheduler

→ 训练

→ 验证

→ 测试

#### 3. 准备工作：导入库

##### 3.1 需要的库
先导入我们这次会用到的核心库：

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

##### 3.2 这些库分别做什么
* torch：张量与训练基础
* torch.nn：定义神经网络层
* torch.optim：优化器
* DataLoader：按 batch 加载数据
* random_split：划分训练集和验证集
* datasets：读取 CIFAR-10
* transforms：做图像预处理与增强

#### 4. 数据准备：下载 CIFAR-10
这次我们使用的是 CIFAR-10，所以可以直接通过 torchvision.datasets.CIFAR10 下载。

##### 4.1 先定义数据保存目录
这表示我们把数据集保存在当前项目下的 cifar-10_data 文件夹中。

In [2]:
data_dir = "./cifar-10_data"

##### 4.2 训练集与测试集读取代码

In [3]:
train_full_dataset = datasets.CIFAR10(
    root=data_dir, 
    train=True,
    download=True,
    transform=None # 后续会添加数据增强
)
test_dataset = datasets.CIFAR10(
    root=data_dir,
    train=False,
    download=True,
    transform=None
)

100%|██████████| 170M/170M [02:05<00:00, 1.36MB/s] 


Extracting ./cifar-10_data/cifar-10-python.tar.gz to ./cifar-10_data
Files already downloaded and verified


##### 4.3 这里为什么先写 transform=None
因为我们后面要先把训练集拆分成：
* train dataset
* val dataset

所以本步骤只是为了读取完整数据集之后，计算一个拆分的索引切片

后续会进行拆分，再分别绑定不同的 transform，得到完整的带有 transform 的可用数据集

这也是更规范的写法。✅

#### 5. 拆分训练集和验证集

##### 5.1 为什么要从训练集里切出验证集
CIFAR-10 默认只提供：
* 训练集
* 测试集

所以验证集通常需要我们自己从训练集中划分出来。

##### 5.2 定义拆分比例
这里我们采用一个非常常见的方案：
* 训练集：80%
* 验证集：20%

因为 CIFAR-10 训练集总共有 50,000 张图像，

所以这里可以划分为：
* 训练集：40,000
* 验证集：10,000

##### 5.3 拆分代码

In [4]:
train_size = int(0.8 * len(train_full_dataset))
val_size = len(train_full_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_full_dataset,
    [train_size, val_size]
)

##### 5.4 这里有一个关键问题
虽然 train_subset 和 val_subset 已经分出来了，

但它们仍然引用的是原来的 train_full_dataset，而原来的 transform 还是 None。

所以我们接下来需要把它们重新包装一下，

让 train 和 val 分别使用不同的 transform。

#### 6. 图像预处理：transforms 配置

##### 6.1 为什么训练集和验证集要分开设置
因为训练集和验证 / 测试集的目标不同：
* 训练集：帮助模型学习，所以可以加入随机增强
* 验证 / 测试集：用于客观评估，所以通常只做基础预处理

因此，三者的 transform 一般不会完全一样。

##### 6.2 训练集 transforms

In [5]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # 因为这里的图像是RGB的，所以有三个通道，因此均值和标准差也有三个值
])

##### 6.3 测试集/验证集 transforms

In [6]:
test_val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

##### 6.4 这些预处理分别在做什么
**（1）RandomHorizontalFlip**

随机水平翻转，增强模型对左右变化的适应能力。

**（2）RandomCrop(32, padding=4)**

先在图像周围补一点边，再随机裁成 32 × 32。

这样可以制造轻微位置变化，提高泛化能力。

**（3）ToTensor**

把 PIL 图像转换成张量，并把像素值从 0~255 转成 0~1。

**（4）Normalize**

把数据进一步归一化，让训练更稳定。

#### 7. 为 train / val / test 绑定不同的 transform

##### 7.1 为什么不能直接给 subset 改 transform

因为 random_split() 返回的是 Subset，

它本身只是`“索引切片”`，不是一个全新的 `dataset` 类。

所以更规范的做法是：

重新创建带 transform 的 CIFAR-10 数据集，再用同样的索引去包装。

##### 7.2 重新创建数据集

In [7]:
train_dataset_with_transform = datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=False,
    transform=train_transform
)

val_dataset_with_transform = datasets.CIFAR10(
    root=data_dir,
    train=True,
    download=False,
    transform=test_val_transform
)

test_dataset = datasets.CIFAR10(
    root=data_dir,
    train=False,
    download=False,
    transform=test_val_transform
)

##### 7.3 用原来的索引重新封装

In [8]:
from torch.utils.data import Subset
train_dataset = Subset(train_dataset_with_transform, train_dataset.indices)
val_dataset = Subset(val_dataset_with_transform, val_dataset.indices)

##### 7.4 这样做的结果
现在我们就得到了三个真正可用的数据集：
* train_dataset：带训练增强
* val_dataset：只做基础预处理
* test_dataset：只做基础预处理

这一步非常重要，因为它体现了：

训练和评估应该使用不同的数据处理策略。

#### 8. 构建 DataLoader

##### 8.1 DataLoader 的作用
DataLoader 的作用是：
* 按 batch 读取数据
* 打乱训练集顺序
* 帮助我们一批一批地送入模型

##### 8.2 配置 batch_size
这里我们先使用一个比较常见的 batch size：

In [9]:
batch_size = 64

##### 8.3 DataLoader 代码

In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
)

##### 8.4 为什么只有 train_loader 要 shuffle
因为：
* 训练集：需要打乱，减少顺序偏差
* 验证集 / 测试集：不需要打乱，因为只是评估

这也是标准做法✅ 

#### 9. 模型构建：一个基础 CNN

##### 9.1 本案例模型思路
这一次，我们不再使用最经典的：

`Conv → ReLU → MaxPool → Conv → ReLU → MaxPool → Flatten → FC`

而是改为一个更现代一点的 CNN 结构思路：
```
Conv → BatchNorm → ReLU
→ Conv(stride=2) → BatchNorm → ReLU
→ Conv(stride=2) → BatchNorm → ReLU
→ GAP
→ Linear
```
这个结构有两个关键变化：

**（1）不使用池化层**

前面我们学过，在现代 CNN 中，中间的下采样不一定必须通过池化层完成。

如果卷积层本身设置：

`stride = 2`

那么它在提取特征的同时，就可以直接把空间尺寸缩小。

也就是说，它可以同时完成：
* 特征提取
* 下采样

所以这一次，我们用：

stride > 1 的卷积层代替池化层

---

**（2）使用 GAP 代替 Flatten**

在传统 CNN 中，卷积层输出通常会先：
* Flatten
* 再进入全连接层

而这次我们不再使用“大 Flatten”，

而是直接在最后接：

`Global Average Pooling（GAP）`

它会把每个通道的整张特征图压缩成一个值，

于是：
* 原来是 [B, C, H, W]
* GAP 后变成 [B, C, 1, 1]
* 再整理成 [B, C]
* 最后直接进入线性分类层

---

**（3）加入 BatchNorm**

为了让训练更稳定，我们在卷积层后加入：

`BatchNorm2d`

所以每一个卷积块都采用：

`Conv → BatchNorm → ReLU`

这是非常常见的现代写法。

##### 9.2 模型代码

In [27]:
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 特征提取部分
        self.features = nn.Sequential(
            # 输入: [B, 3, 32, 32]
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # 输出： [B, 32, 32, 32]
            # 下采样 1：32x32 -> 16x16
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # 输出： [B, 64, 16, 16]
            # 下采样 2：16x16 -> 8x8
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # 输出： [B, 128, 8, 8]
            # 再加上一层特征提取，不改变形状
            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # 输出： [B, 128, 8, 8]
        )
        # 全局平均池化
        self.gap = nn.AdaptiveAvgPool2d((1, 1)) # 输出 [B, 128, 1, 1]
        # 分类器
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), # 因为全局平均池化后特征图的通道数是128，所以全连接层的输入特征数是128
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(64, 10) # 因为CIFAR-10有10个类别，所以输出特征数是10
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1) # 将 [B, 128, 1, 1] 展平为 [B, 128]
        x = self.classifier(x)
        return x

##### 9.3 shape 变化解释
这一部分非常重要，因为现代 CNN 中虽然没有池化层和大 Flatten，

但我们仍然要清楚每一层 shape 是怎么变化的。

**（1）输入图像**

CIFAR-10 的输入图像 shape 是：

`[B, 3, 32, 32]`

其中：
* B：batch size
* 3：RGB 三通道
* 32 × 32：图像大小

---

**（2）第一层卷积**

`nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)`

这一层：
* 不改变空间尺寸
* 只把通道数从 3 变成 32

所以输出变成：

`[B, 32, 32, 32]`

---

**（3）第二层卷积（stride=2）**

`nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)`

这一层开始进行下采样：
* 空间尺寸：32 × 32 → 16 × 16
* 通道数：32 → 64

所以输出变成：

`[B, 64, 16, 16]`

这里你可以看到：

`stride = 2` 已经起到了传统池化层的作用。

---

**（4）第三层卷积（stride=2）**

`nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)`

继续下采样：
* 空间尺寸：16 × 16 → 8 × 8
* 通道数：64 → 128

所以输出变成：

`[B, 128, 8, 8]`

---

**（5）第四层卷积（stride=1）**

`nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1)`

这一层不做下采样，只继续提取更高层特征：
* 空间尺寸保持 8 × 8
* 通道数保持 128

输出仍然是：

`[B, 128, 8, 8]`

---

**（6）GAP 全局平均池化**

`nn.AdaptiveAvgPool2d((1, 1))`

这一层会把每个通道的一整张 8 × 8 特征图压缩成一个值：
* 空间尺寸：8 × 8 → 1 × 1
* 通道数保持不变：128

所以输出变成：

`[B, 128, 1, 1]`

---

**（7）去掉多余空间维度**

`torch.flatten(x, 1)`

这里虽然也写了 flatten，

但它已经不是传统意义上的“大 Flatten”了。

它只是把：

`[B, 128, 1, 1] → [B, 128]`

也就是说，这里只是把 GAP 后多余的 1 × 1 空间维去掉，

方便送入最后的线性层。

---

**（8）最终分类层**
``` python
nn.Linear(128, 64), # 因为全局平均池化后特征图的通道数是128，所以全连接层的输入特征数是128
nn.ReLU(), 
nn.Dropout(0.3),
nn.Linear(64, 10) # 因为CIFAR-10有10个类别，所以输出特征数是10
```
最后：
* 输入 128 维特征
* 输出 10 个类别分数

所以最终输出 shape 是：

`[B, 10]`

这对应 CIFAR-10 的 10 个类别。

#### 10. 准备训练配置

##### 10.1 设备选择
如果有 GPU，就优先用 GPU；否则用 CPU。

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

##### 10.2  实例化模型

In [29]:
model = CIFAR10CNN().to(device)

##### 10.3 损失函数

In [30]:
criterion = nn.CrossEntropyLoss()

##### 10.4 优化器
这里我们先使用：

In [31]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

##### 10.5 scheduler 学习率衰减

In [32]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

/home/zhang/miniconda3/envs/da/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


#### 11. 训练函数

##### 11.1 为什么要单独写训练函数
因为每个 epoch 的训练逻辑都一样，

如果每次都手写，会很重复。

所以我们把训练一个 epoch 的过程封装成函数

##### 11.2 训练函数代码

In [33]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

##### 11.3 这里的关键步骤解释
**（1）model.train()**

告诉 PyTorch：当前处于训练模式。

这样 Dropout 会启用。

---

**（2）optimizer.zero_grad()**

每个 batch 训练前先清空旧梯度。

---

**（3）outputs = model(images)**

前向传播。

---

**（4）loss.backward()**

反向传播，计算梯度。

---

**（5）optimizer.step()**

更新参数。


#### 12. 验证函数

##### 12.1 为什么验证要单独写
因为验证过程和训练过程不同：
* 不需要反向传播
* 不需要更新参数
* 只需要前向计算并评估结果

##### 12.2 验证函数代码

In [34]:
def evaluate(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

##### 12.3 这里最关键的两个点
**（1）model.eval()**

切换到评估模式。

这样 Dropout 会关闭，BatchNorm（如果有）也会切到评估行为。

---

**（2）torch.no_grad()**

告诉 PyTorch：这里不用计算梯度。

这样更省显存、更快。

#### 13. 完整训练循环

In [35]:
epochs = 30

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print(f"Current LR: {current_lr:.6f}")
    print("-" * 50)

Epoch [1/30]
Train Loss: 1.6525 | Train Acc: 0.3835
Val   Loss: 1.3414 | Val   Acc: 0.4971
Current LR: 0.001000
--------------------------------------------------
Epoch [2/30]
Train Loss: 1.3416 | Train Acc: 0.5173
Val   Loss: 1.1544 | Val   Acc: 0.5792
Current LR: 0.001000
--------------------------------------------------
Epoch [3/30]
Train Loss: 1.2143 | Train Acc: 0.5664
Val   Loss: 1.0193 | Val   Acc: 0.6337
Current LR: 0.001000
--------------------------------------------------
Epoch [4/30]
Train Loss: 1.1235 | Train Acc: 0.6007
Val   Loss: 1.0093 | Val   Acc: 0.6314
Current LR: 0.001000
--------------------------------------------------
Epoch [5/30]
Train Loss: 1.0643 | Train Acc: 0.6239
Val   Loss: 0.9279 | Val   Acc: 0.6671
Current LR: 0.001000
--------------------------------------------------
Epoch [6/30]
Train Loss: 1.0149 | Train Acc: 0.6422
Val   Loss: 1.0045 | Val   Acc: 0.6370
Current LR: 0.001000
--------------------------------------------------
Epoch [7/30]
Train Los

##### 13.1 个训练循环的逻辑
每个 epoch 都会做：
* 用训练集跑一遍 → 更新参数
* 用验证集跑一遍 → 观察泛化表现
* scheduler 调整学习率
* 打印当前结果

这就是最标准的 train + val 训练流程。

#### 14. 最终测试评估

##### 14.1 为什么测试要放到最后
因为测试集应该只在模型训练完成之后使用。

如果在训练过程中频繁看测试集结果，就会破坏测试集的“最终评估”意义。

##### 14.2 测试代码

In [36]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print("Final Test Result")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")

Final Test Result
Test Loss: 0.5799
Test Acc : 0.8022


##### 14.3 测试阶段在看什么
测试阶段主要看的是：
* 模型在未见过数据上的 loss
* 模型在未见过数据上的 accuracy

这是我们最终最关心的结果。